In [0]:
# Definição do catalogo, schema e volume
catalog_name = "cinedata_analytics"
schema_name = "tabelas"
volume_name = "inputs"

# Criação do catalogo e schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

In [0]:
from pyspark.sql.functions import current_timestamp

# Copia os arquivos para o volume
dbutils.fs.cp(
    "file:/Workspace/Users/severojoaoopedro90@gmail.com/projeto-cinedata-analytics/Inputs - Atividade Engenharia de Dados - Bases de Dados/",
    f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/",
    recurse=True # Recursividade
)

# Criação do schema bronze
schema_bronze = "bronze"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_bronze}")

base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/" # Definindo o caminho

# Criação de listas com os arquivos csvs
arquivos = [ 
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

for arquivo in arquivos:
    nome_tabela = arquivo.replace(".csv","")
    df = spark.read.csv( # Lendo os arquivos
        base_path+arquivo,
        header=True, # Definindo o cabeçalho
        inferSchema=True # Inferindo o schema 
    )
    
    # Adicionando uma coluna com a data de ingestão
    df = df.withColumn("ingestion_datetime", current_timestamp()) 

    # Salvando os arquivos no schema bronze
    df.write.mode("append").format("delta").saveAsTable(f"{catalog_name}.{schema_bronze}.{nome_tabela}") 
    print(f"Tabela {nome_tabela} criada com sucesso.")